# Fake News Detection (improved)
Run the cells top to bottom. Sections marked **OPTIONAL** can be skipped.

## 1. Setup

In [ ]:
%pip install -q pandas numpy scikit-learn joblib matplotlib

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

RANDOM_STATE = 42

## 2. Load data, label, remove empty and duplicate rows

In [ ]:
df_fake = pd.read_csv('Fake.csv'); df_fake['label'] = 0   # 0 = Fake
df_true = pd.read_csv('True.csv'); df_true['label'] = 1   # 1 = Real

df = pd.concat([df_fake, df_true], ignore_index=True)
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['date'] = pd.to_datetime(df['date'], errors='coerce')   # used only in the optional time-split section

# Drop empty and duplicate articles BEFORE splitting (prevents train/test leakage)
before = len(df)
df = df[df['content'].str.strip().str.len() > 20]
df = df.drop_duplicates(subset='content').reset_index(drop=True)
print(f"Removed {before - len(df)} empty/duplicate rows -> {len(df)} remain")
print(df['label'].value_counts())

## 3. Clean text and split

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'^.{0,120}?\(reuters\)\s*-?\s*', '', text, flags=re.I)  # "WASHINGTON (Reuters) -"
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)   # URLs
    text = re.sub(r'<.*?>', ' ', text)                    # HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)                 # punctuation and digits -> space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned'] = df['content'].apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label'],
    test_size=0.20, random_state=RANDOM_STATE, stratify=df['label']
)
print(len(X_train), "train /", len(X_test), "test")

## 4. Define models (each has its own TF-IDF inside a Pipeline)

In [ ]:
stop_words = list(ENGLISH_STOP_WORDS | {'reuters', 'said', 'image', 'featured', 'getty', 'via'})

def make_pipeline(clf):
    return Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                  stop_words=stop_words, min_df=3, max_df=0.9,
                                  sublinear_tf=True)),
        ('clf', clf),
    ])

models = {
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=1000, C=5)),
    'Linear SVM':          make_pipeline(LinearSVC(C=1.0)),
    'Passive Aggressive':  make_pipeline(PassiveAggressiveClassifier(max_iter=1000, random_state=RANDOM_STATE)),
    'Multinomial NB':      make_pipeline(MultinomialNB()),
}

## 5. Compare models with 5-fold CV on the TRAIN set only

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring=['accuracy', 'f1'], n_jobs=-1)
    results[name] = {
        'CV Accuracy': scores['test_accuracy'].mean(),
        'CV F1': scores['test_f1'].mean(),
        'Std (F1)': scores['test_f1'].std(),
    }

results_df = pd.DataFrame(results).T.sort_values('CV F1', ascending=False)
print(results_df.round(4))

best_name = results_df.index[0]
baseline_cv_f1 = results_df.loc[best_name, 'CV F1']
print(f"\nBest model by CV F1: {best_name} ({baseline_cv_f1:.4f})")

## 6. Fit the best model and evaluate ONCE on the test set

In [ ]:
best_pipe = models[best_name]
best_pipe.fit(X_train, y_train)
preds = best_pipe.predict(X_test)

print(classification_report(y_test, preds, target_names=['Fake', 'Real'], digits=4))
ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=['Fake', 'Real'], cmap='Blues')
plt.title(f'{best_name} - Test Set')
plt.show()

## 7. What did the model learn? (linear models)

In [ ]:
tfidf = best_pipe.named_steps['tfidf']
clf = best_pipe.named_steps['clf']

if hasattr(clf, 'coef_'):
    feats = np.array(tfidf.get_feature_names_out())
    coefs = clf.coef_.ravel()
    print("Strongest FAKE indicators:", list(feats[np.argsort(coefs)[:20]]))
    print("Strongest REAL indicators:", list(feats[np.argsort(coefs)[-20:]]))

## 8. Save the baseline model

In [ ]:
joblib.dump(best_pipe, 'fake_news_pipeline_baseline.pkl')
print("Saved baseline model.")

## 9. Error analysis (look at the mistakes)

In [ ]:
mask = preds != y_test.values
err_df = df.loc[X_test.index[mask], ['title', 'label']].copy()
err_df['predicted'] = preds[mask]
err_df['true'] = err_df['label'].map({0: 'Fake', 1: 'Real'})
err_df['predicted'] = err_df['predicted'].map({0: 'Fake', 1: 'Real'})
print(f"{mask.sum()} errors out of {len(y_test)}")
print(err_df[['title', 'true', 'predicted']].head(15).to_string())

# Are errors linked to article length?
df['word_count'] = df['cleaned'].str.split().str.len()
print("\nMean words, all test articles :", round(df.loc[X_test.index, 'word_count'].mean(), 1))
print("Mean words, misclassified     :", round(df.loc[X_test.index[mask], 'word_count'].mean(), 1))

## 10. Decision threshold check (catch more fakes vs. false alarms)

In [ ]:
if hasattr(best_pipe, 'decision_function'):
    scores = best_pipe.decision_function(X_test)
    yt = y_test.values
    rows = []
    for t in [-0.5, -0.25, 0, 0.25, 0.5]:
        p = (scores > t).astype(int)
        rows.append({
            'threshold': t,
            'fake recall': round(((p == 0) & (yt == 0)).sum() / (yt == 0).sum(), 4),
            'real recall': round(((p == 1) & (yt == 1)).sum() / (yt == 1).sum(), 4),
        })
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("This model has no decision_function; skip this section.")

## 11. Hyperparameter tuning (takes a few minutes)

In [ ]:
tune_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=stop_words, sublinear_tf=True, min_df=3, max_df=0.9)),
    ('clf', LinearSVC()),
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [20000, 50000],
    'clf__C': [0.1, 0.5, 1, 2],
}

grid = GridSearchCV(tune_pipe, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Tuned CV F1   :", round(grid.best_score_, 4))
print("Baseline CV F1:", round(baseline_cv_f1, 4))

# Choose using CV score only (not the test set)
if grid.best_score_ > baseline_cv_f1:
    final_pipe, final_name = grid.best_estimator_, 'Tuned Linear SVM'
else:
    final_pipe, final_name = best_pipe, best_name
print("Final model:", final_name)

final_preds = final_pipe.predict(X_test)
print(classification_report(y_test, final_preds, target_names=['Fake', 'Real'], digits=4))
ConfusionMatrixDisplay.from_predictions(y_test, final_preds, display_labels=['Fake', 'Real'], cmap='Blues')
plt.title(f'{final_name} - Test Set')
plt.show()

## 12. Save the final model and add a prediction helper

In [ ]:
joblib.dump(final_pipe, 'fake_news_pipeline.pkl')   # this is the file app.py loads
print("Saved fake_news_pipeline.pkl")

def predict_news(text, pipeline=None):
    pipeline = pipeline or final_pipe
    cleaned = clean_text(text)
    label = pipeline.predict([cleaned])[0]
    clf = pipeline.named_steps['clf']
    if hasattr(clf, 'predict_proba'):
        conf = pipeline.predict_proba([cleaned])[0].max()
    else:
        conf = 1 / (1 + np.exp(-abs(pipeline.decision_function([cleaned])[0])))
    return ('Real' if label == 1 else 'Fake'), round(float(conf), 3)

print(predict_news("Scientists announce breakthrough in battery technology at university lab."))

## 13. Out-of-domain test (most honest accuracy number)
Replace the examples with your own 20-50 headlines or short articles from other topics/years. Label 1 = real, 0 = fake.

In [ ]:
my_tests = [
    ("The central bank held interest rates steady on Wednesday and said inflation was easing.", 1),
    ("The national cricket team won the series after a strong bowling performance in the final match.", 1),
    ("Researchers report a new method for recycling lithium batteries at lower cost.", 1),
    ("City council approves funding for a new metro line after months of public consultation.", 1),
    ("SHOCKING: doctors hate this one trick that cures every disease overnight, share before it is deleted!", 0),
    ("Secret document proves the moon landing was filmed in a basement, the media is hiding it from you.", 0),
    ("Celebrity secretly replaced by a clone, insiders reveal the truth that elites do not want you to know.", 0),
    ("Government puts mind control chips in tap water, whistleblower says, wake up people.", 0),
]

ood = pd.DataFrame(my_tests, columns=['text', 'true'])
ood['pred'] = [1 if predict_news(t)[0] == 'Real' else 0 for t in ood['text']]
print(ood[['true', 'pred']].T.to_string())
print("Out-of-domain accuracy:", round((ood['true'] == ood['pred']).mean(), 3))

## 14. OPTIONAL: time-based split (train on earlier articles, test on later ones)

In [ ]:
df_t = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
cut = int(len(df_t) * 0.8)
Xt_train, Xt_test = df_t['cleaned'][:cut], df_t['cleaned'][cut:]
yt_train, yt_test = df_t['label'][:cut], df_t['label'][cut:]
print("Train up to:", df_t['date'][cut - 1].date(), "| Test from:", df_t['date'][cut].date())

time_pipe = clone(final_pipe)
time_pipe.fit(Xt_train, yt_train)
print(classification_report(yt_test, time_pipe.predict(Xt_test), target_names=['Fake', 'Real'], digits=4))

## 15. OPTIONAL: word + character n-grams (slower)

In [ ]:
features = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), stop_words=stop_words, min_df=3, sublinear_tf=True)),
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5,
                             sublinear_tf=True, max_features=100000)),
])
pipe_union = Pipeline([('features', features), ('clf', LinearSVC(C=1))])

cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
s = cross_validate(pipe_union, X_train, y_train, cv=cv3, scoring='f1', n_jobs=1)
print("Word+char CV F1:", round(s['test_score'].mean(), 4), "| Baseline CV F1:", round(baseline_cv_f1, 4))